# Лабораторна робота 3. Машинне навчання
## Регресія: прогнозування вартості нерухомості

**Мета:** первинний аналіз даних, побудова кількох моделей регресії, оцінка якості та підбір гіперпараметрів Random Forest.

**Датасет:** https://www.kaggle.com/datasets/prokshitha/home-value-insights

## 1. Імпорт бібліотек

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 2. Завантаження даних

In [ ]:
# Варіант 1 — з комп’ютера (Colab)
# from google.colab import files
# uploaded = files.upload()
# df = pd.read_csv(list(uploaded.keys())[0])

In [ ]:
# Варіант 2 — KaggleHub
import kagglehub
import os

path = kagglehub.dataset_download('prokshitha/home-value-insights')
print('Path:', path)
# Знайдіть потрібний csv у папці
print(os.listdir(path))
df = pd.read_csv(os.path.join(path, os.listdir(path)[0]))  # або вкажіть точну назву файлу
df.head()

## 3. Опис ознак
- Square_Footage, Num_Bedrooms, Num_Bathrooms, Year_Built, Lot_Size, Garage_Size, Neighborhood_Quality, House_Price (цільова)

## 4. Первинний аналіз даних

In [ ]:
print('Розмір:', df.shape)
display(df.head())
df.info()

In [ ]:
print('Дублікати:', df.duplicated().sum())

In [ ]:
print(df.isnull().sum())

In [ ]:
print(df.describe())

## 5. Аналіз окремих ознак

In [ ]:
print(df['Neighborhood_Quality'].value_counts())

In [ ]:
sns.countplot(x='Neighborhood_Quality', data=df)
plt.title('Розподіл Neighborhood_Quality')
plt.show()

In [ ]:
print(df['Num_Bathrooms'].value_counts())

In [ ]:
sns.countplot(x='Num_Bathrooms', data=df)
plt.title('Розподіл Num_Bathrooms')
plt.show()

In [ ]:
# Додатково — pairplot (може бути довгим)
# sns.pairplot(df.sample(min(500, len(df))), diag_kind='kde')
# plt.show()

## 6. Кореляційний аналіз

In [ ]:
corr = df.corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Кореляційна матриця')
plt.tight_layout()
plt.show()

In [ ]:
print(corr['House_Price'].sort_values(ascending=False))

## 7. Побудова моделей регресії

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [ ]:
X = df.drop(columns=['House_Price'])
y = df['House_Price']
print(X.shape, y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(X_train.shape, X_test.shape)

In [ ]:
models = {
    'Linear Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LinearRegression())
    ]),
    'Ridge': Pipeline([
        ('scaler', StandardScaler()),
        ('model', Ridge())
    ]),
    'Lasso': Pipeline([
        ('scaler', StandardScaler()),
        ('model', Lasso())
    ]),
    'Random Forest': RandomForestRegressor(random_state=42, n_estimators=100),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42)
}

In [ ]:
results = []
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[name] = y_pred
    results.append({
        'Model': name,
        'R2': r2_score(y_test, y_pred),
        'MAE': mean_absolute_error(y_test, y_pred),
        'MSE': mean_squared_error(y_test, y_pred)
    })

results_df = pd.DataFrame(results).sort_values('R2', ascending=False)
print(results_df)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for i, (name, y_pred) in enumerate(predictions.items()):
    ax = axes[i]
    ax.scatter(y_test, y_pred, alpha=0.5)
    mn = min(y_test.min(), y_pred.min())
    mx = max(y_test.max(), y_pred.max())
    ax.plot([mn, mx], [mn, mx], 'r--')
    ax.set_xlabel('Фактична ціна')
    ax.set_ylabel('Прогноз')
    ax.set_title(name)

if len(predictions) < 6:
    axes[-1].axis('off')
plt.tight_layout()
plt.show()

## 8. Підбір гіперпараметрів Random Forest

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_features': ['sqrt', 'log2', None],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

grid_rf = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1
)
grid_rf.fit(X_train, y_train)

print('Найкращі параметри:', grid_rf.best_params_)
print('CV R2:', grid_rf.best_score_)

In [ ]:
best_rf = grid_rf.best_estimator_
y_pred_opt = best_rf.predict(X_test)

print('Оптимізований RF:')
print('R2 :', r2_score(y_test, y_pred_opt))
print('MAE:', mean_absolute_error(y_test, y_pred_opt))
print('MSE:', mean_squared_error(y_test, y_pred_opt))

In [ ]:
base_rf = results_df[results_df['Model'] == 'Random Forest'].iloc[0]
print('Порівняння RF:')
print(f"Базовий   → R2={base_rf['R2']:.4f}, MAE={base_rf['MAE']:.2f}, MSE={base_rf['MSE']:.2f}")
print(f"Оптиміз.  → R2={r2_score(y_test, y_pred_opt):.4f}, MAE={mean_absolute_error(y_test, y_pred_opt):.2f}, MSE={mean_squared_error(y_test, y_pred_opt):.2f}")

**Висновок щодо оптимізації:**  
Підбір гіперпараметрів зазвичай покращує R² і зменшує помилки. Оптимізовану модель доцільно використовувати.

## 9. Аналіз прогнозів лінійної регресії

In [ ]:
lr_model = models['Linear Regression']
y_pred_lr = predictions['Linear Regression']

table = pd.DataFrame({
    'Фактична ціна': y_test.values,
    'Прогноз': y_pred_lr,
    'Похибка %': np.abs(y_test.values - y_pred_lr) / y_test.values * 100
})
print(table.sample(10, random_state=42))

## 10. Висновки

1. Побудовано 5 моделей: Linear, Ridge, Lasso, Random Forest, Gradient Boosting.
2. Найкращу якість зазвичай показує Random Forest або Gradient Boosting (найвищий R², менші MAE/MSE).
3. Підбір гіперпараметрів RF покращує метрики.
4. Прогнози на тестовій вибірці достатньо точні (залежить від даних).
5. Найбільший вплив на ціну зазвичай мають площа, кількість спалень/ванн, якість району.